In [1]:
import pdfplumber
from pathlib import Path
from pdfplumber.utils import extract_text
import re

In [2]:
ROOT = Path("~/code/selector_examenes").expanduser()

In [3]:
file = ROOT / Path("examenes_quimica/2026_quimica/2026 - quimica - ordinaria titular titular quínica.pdf")

In [4]:
# Ya no hacen falta HEADER/INSTRUCTIONS hardcodeados: se eliminan por regex / por el split.

In [5]:
SUB = str.maketrans("0123456789+-–", "₀₁₂₃₄₅₆₇₈₉₊₋₋")
SUP = str.maketrans("0123456789+-–", "⁰¹²³⁴⁵⁶⁷⁸⁹⁺⁻⁻")

In [6]:
def page_text(page, small=8):
    """Texto de la página convirtiendo los caracteres pequeños en sub/superíndices Unicode."""
    chars, prev = [], None
    for c in sorted(page.chars, key=lambda c: (round(c["top"]), c["x0"])):
        if c["size"] < small and prev is not None:
            # superíndice si su base queda claramente por encima de la del texto normal
            table = SUP if c["bottom"] < prev["bottom"] - 2 else SUB
            c = dict(c, text=c["text"].translate(table))
        else:
            prev = c
        chars.append(c)
    return extract_text(chars, y_tolerance=4)


def split_questions(path):
    with pdfplumber.open(path) as pdf:
        text = "\n".join(page_text(p) for p in pdf.pages)
    # cabecera repetida en cada página
    text = re.sub(r"^PRUEBA DE ACCESO.*?CURSO \d{4}-\d{4}\n", "", text, flags=re.S | re.M)
    # lo anterior a la PREGUNTA 1 (instrucciones y constantes) cae en el primer trozo y se descarta
    return re.split(r"(?m)^(?=PREGUNTA \d+\.-)", text)[1:]

In [7]:
questions = split_questions(file)

In [8]:
len(questions)

5

In [13]:
print(*questions, sep="\n" + "-" * 10 + "\n\n")

PREGUNTA 1.- (2 puntos: 0,5 puntos cada apartado). Responda a UNA de las siguientes cuestiones (1A o 1B).
1A. Consideres los elementos cuyos números atómicos son 11 y 16. Conteste a las siguientes cuestiones:
a) Escriba sus configuraciones electrónicas.
b) Basándose en la configuración electrónica, explique el grupo y periodo donde se encuentra cada elemento.
c) Justifique qué elemento tiene mayor radio atómico.
d) Razone cuál es el ion más estable que forma cada elemento.
1B. Justifique si son verdaderas o falsas las siguientes afirmaciones:
a) El CH₄ tiene un punto de fusión menor que el CCl₄
b) El CCl₄ no es soluble en agua.
c) El átomo central del BF₃ presenta una hibridación sp³
d) El BF₃ es un buen conductor de la electricidad.

----------

PREGUNTA 2.- (2 puntos: 0,5 puntos cada apartado). Responda a UNA de las siguientes cuestiones (2A o 2B).
2A. Justifique la espontaneidad de cada uno de los siguientes procesos en función de la temperatura:
a) CaCO₃(s) → CaO(s) + CO₂(g) ∆H> 0
